# Qwen3.5-9B-AWQ typed full LLM multi-agent TEC model-ablation experiment

This notebook runs the same typed-contract full LLM multi-agent network as `04_qwen_multi_agent_typed_colab.ipynb`, but swaps the model to `QuantTrio/Qwen3.5-9B-AWQ`. The architecture, prompts, tools, dataset, five-task benchmark, GoldRunner evaluation, and metrics are intentionally unchanged. The only experimental variable is replacing `Qwen/Qwen3.5-4B` with the pre-quantized AWQ checkpoint of `Qwen/Qwen3.5-9B`.


## 1. Clean Colab setup

Run this in Google Colab first. After it finishes, restart the runtime before importing transformers. This notebook expects an already processed TEC parquet and does not rebuild IONEX data.

This model-ablation notebook uses a pre-quantized AWQ checkpoint. It does not perform on-the-fly bitsandbytes quantization and does not download the base BF16/FP16 `Qwen/Qwen3.5-9B` weights for inference.


In [1]:
# # Clean Colab setup for Qwen3.5-9B-AWQ.
# # Run this cell first, then restart runtime before importing transformers.
# # The checkApoint is already AWQ-quantized, so no bitsandbytes quantization is used.

# !pip uninstall -y transformers scikit-learn scipy
# !pip install -U accelerate torchvision pillow sentencepiece protobuf safetensors huggingface_hub pandas pyarrow pydantic python-dateutil
# !pip install gptqmodel
# !pip install -U "transformers @ git+https://github.com/huggingface/transformers.git@f2ba019"


In [2]:
# ============================================================================
# Colab setup for pre-quantized Qwen3.5-9B BitsAndBytes 4-bit
# with CUDA 12.8-compatible PyTorch.
#
# Model:
#   lainlives/Qwen3.5-9B-bnb-4bit
#
# Why this is needed:
# - The model loads successfully as a pre-quantized BNB 4-bit checkpoint.
# - bitsandbytes fails during the first forward pass when PyTorch is built
#   against CUDA 13.x because Colab lacks libnvJitLink.so.13.
# - We therefore pin PyTorch to a CUDA 12.8 build.
#
# Do NOT install:
#   gptqmodel, autoawq, auto-gptq
#
# Do NOT use:
#   BitsAndBytesConfig(load_in_4bit=True)
# ============================================================================

import sys
import subprocess

PYTHON = sys.executable

print("Installing CUDA 12.8 PyTorch stack...")

# Install a CUDA 12.8 PyTorch build instead of the problematic cu130 build.
subprocess.run(
    [
        PYTHON, "-m", "pip", "install", "-q", "-U", "--force-reinstall",
        "torch==2.10.0",
        "torchvision==0.25.0",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ],
    check=True,
)

print("Installing inference and project dependencies...")

subprocess.run(
    [
        PYTHON, "-m", "pip", "install", "-q", "-U",
        "accelerate>=1.0.0",
        "bitsandbytes>=0.49.1",
        "safetensors>=0.4.5",
        "huggingface_hub>=0.28.0",
        "sentencepiece",
        "protobuf",
        "pillow",
        "pandas",
        "pyarrow",
        "pydantic>=2.0",
        "python-dateutil",
    ],
    check=True,
)

print("Installing Transformers without changing the CUDA/PyTorch stack...")

# Transformers 5.9.0 already recognized qwen3_5 in your previous attempt.
subprocess.run(
    [
        PYTHON, "-m", "pip", "install", "-q", "-U",
        "--no-deps",
        "transformers==5.9.0",
    ],
    check=True,
)

# transformers 5.9.0 rejects tokenizers 0.23.1;
# the available <=0.23.0 PyPI build is the release candidate.
subprocess.run(
    [
        PYTHON, "-m", "pip", "install", "-q", "-U", "--force-reinstall",
        "--no-deps",
        "tokenizers==0.23.0rc0",
    ],
    check=True,
)

print()
print("Installation completed.")
print("Now restart the Colab session before importing torch or transformers:")
print("Runtime -> Restart session")

Installing CUDA 12.8 PyTorch stack...
Installing inference and project dependencies...
Installing Transformers without changing the CUDA/PyTorch stack...

Installation completed.
Now restart the Colab session before importing torch or transformers:
Runtime -> Restart session


IMPORTANT: after setup, restart the Colab runtime before importing transformers.

## 2. Import check

In [3]:
import torch
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.10.0+cu128
transformers: 5.9.0
cuda: True
gpu: Tesla T4


## 3. Clone or update repository

In [4]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "main"], check=True)

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print("Working directory:", Path.cwd())

Working directory: /content/tec_agent_project


## 4. Project imports

In [5]:
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from tec_agents.agents.llm_multi_agent_typed import LLMFullTypedMultiAgent
from tec_agents.agents.llm_multi_agent_typed_prompts import ARCHITECTURE_NAME
from tec_agents.agents.llm_multi_agent_typed_protocol import TYPED_PROTOCOL_VERSION
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.five_task_configs import (
    get_five_task_configs,
    build_five_task_eval_task,
    build_five_task_expected_sequence,
)
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import MetricResult, compare_agent_to_gold
from tec_agents.eval.task_set import task_to_dict
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server


## 5. CONFIG

In [6]:
# MODEL_NAME = "QuantTrio/Qwen3.5-9B-AWQ"
# BASE_MODEL_NAME = "Qwen/Qwen3.5-9B"
# QUANTIZATION_FORMAT = "AWQ_INT4_PREQUANTIZED"
# MODEL_ABLATION_ID = "qwen35_9b_awq"
# LOAD_IN_4BIT = False
# LOAD_IN_8BIT = False
# TORCH_DTYPE = "float16"
# MAX_INPUT_TOKENS = 4096
# MAX_NEW_TOKENS = 512
# TEMPERATURE = 0.0
# DO_SAMPLE = False

# DATASET_REF = "default"
# DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
# PROCESSED_DATASET_PATH = PROJECT_DIR / "data" / "processed" / DATASET_FILENAME
# START = "2024-03-01"
# END = "2024-04-01"

# ARCHITECTURE_MODE = "qwen_multi_agent_typed_full_llm"
# PROMPT_REVISION = "grounded_inputs_deliverables_single_block_v3"
# EXPERIMENT_ID = "qwen_multi_agent_typed_v3_qwen35_9b_awq_batch_colab"
# MAX_ORCHESTRATION_STEPS = 12
# MAX_ROLE_STEPS = 8
# MAX_TOOL_CALLS_PER_ROLE = 8
# MAX_PARSE_RETRIES = 2
# MAX_TOOL_RETRIES = 2
# STATE_FEEDBACK_MODE = "typed_state"

# RUN_ALL_TASKS = True
# SELECTED_PRESET = "hightec_midlat_europe"

# OUTPUT_ROOT = PROJECT_DIR / "outputs" / "metrics"
# EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / "experiment_5_qwen35_9b_awq"
# OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR
# BATCH_OUTPUT_PATH = EXPERIMENT_OUTPUT_DIR / "qwen_multi_agent_typed_v3_qwen35_9b_awq_batch_colab.json"
# PER_TASK_OUTPUT_TEMPLATE = "qwen_multi_agent_typed_v3_qwen35_9b_awq_{preset_id}_colab.json"
# MIN_FREE_VRAM_GB_AFTER_LOAD = 1.0

# TASK_CONFIGS = get_five_task_configs(dataset_ref=DATASET_REF)
# if RUN_ALL_TASKS:
#     SELECTED_TASK_CONFIGS = TASK_CONFIGS
# else:
#     SELECTED_TASK_CONFIGS = [c for c in TASK_CONFIGS if c["preset_id"] == SELECTED_PRESET]
# assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET={SELECTED_PRESET!r}"
# print("Architecture:", ARCHITECTURE_MODE)
# print("Typed protocol:", TYPED_PROTOCOL_VERSION)
# print("Prompt revision:", PROMPT_REVISION)
# print("Model ablation:", MODEL_ABLATION_ID)
# print("Model:", MODEL_NAME)
# print("Base model:", BASE_MODEL_NAME)
# print("Quantization:", QUANTIZATION_FORMAT)
# print("Output dir:", EXPERIMENT_OUTPUT_DIR)
# print("Selected tasks:", [c["preset_id"] for c in SELECTED_TASK_CONFIGS])


In [7]:
# ============================================================================
# MODEL ABLATION CONFIG: Qwen3.5-9B pre-quantized BNB 4-bit
# Same typed v3 architecture, prompts, tools, metrics and five-task benchmark.
# Only the model checkpoint / loading backend / output namespace are changed.
# ============================================================================

MODEL_NAME = "lainlives/Qwen3.5-9B-bnb-4bit"
BASE_MODEL_NAME = "Qwen/Qwen3.5-9B"
QUANTIZATION_FORMAT = "BNB_NF4_PREQUANTIZED"
MODEL_ABLATION_ID = "qwen35_9b_bnb_4bit"

# Important:
# This checkpoint is already quantized and stored as 4-bit weights.
# Do NOT apply BitsAndBytesConfig(load_in_4bit=True) again in the loading cell.
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False

TORCH_DTYPE = "bfloat16"
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
DO_SAMPLE = False

DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
PROCESSED_DATASET_PATH = PROJECT_DIR / "data" / "processed" / DATASET_FILENAME
START = "2024-03-01"
END = "2024-04-01"

# Architecture and prompts are intentionally unchanged.
ARCHITECTURE_MODE = "qwen_multi_agent_typed_full_llm"
PROMPT_REVISION = "grounded_inputs_deliverables_single_block_v3"

# New model-ablation experiment identity.
EXPERIMENT_ID = "qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_batch_colab"

MAX_ORCHESTRATION_STEPS = 12
MAX_ROLE_STEPS = 8
MAX_TOOL_CALLS_PER_ROLE = 8
MAX_PARSE_RETRIES = 2
MAX_TOOL_RETRIES = 2
STATE_FEEDBACK_MODE = "typed_state"

RUN_ALL_TASKS = True
SELECTED_PRESET = "hightec_midlat_europe"

# Keep this run isolated from the failed AWQ-loading attempt and from 4B runs.
OUTPUT_ROOT = PROJECT_DIR / "outputs" / "metrics"
EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / "experiment_5b_qwen35_9b_bnb_4bit"
OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR

BATCH_OUTPUT_PATH = (
    EXPERIMENT_OUTPUT_DIR
    / "qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_batch_colab.json"
)

PER_TASK_OUTPUT_TEMPLATE = (
    "qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_{preset_id}_colab.json"
)

# Keep the same technical memory safety check used in the AWQ notebook.
MIN_FREE_VRAM_GB_AFTER_LOAD = 1.0

EXPERIMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK_CONFIGS = get_five_task_configs(dataset_ref=DATASET_REF)

if RUN_ALL_TASKS:
    SELECTED_TASK_CONFIGS = TASK_CONFIGS
else:
    SELECTED_TASK_CONFIGS = [
        c for c in TASK_CONFIGS
        if c["preset_id"] == SELECTED_PRESET
    ]

assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET={SELECTED_PRESET!r}"

print("Architecture:", ARCHITECTURE_MODE)
print("Typed protocol:", TYPED_PROTOCOL_VERSION)
print("Prompt revision:", PROMPT_REVISION)
print("Model ablation:", MODEL_ABLATION_ID)
print("Model:", MODEL_NAME)
print("Base model:", BASE_MODEL_NAME)
print("Quantization:", QUANTIZATION_FORMAT)
print("On-the-fly 4-bit quantization:", LOAD_IN_4BIT)
print("Output dir:", EXPERIMENT_OUTPUT_DIR)
print("Selected tasks:", [c["preset_id"] for c in SELECTED_TASK_CONFIGS])

Architecture: qwen_multi_agent_typed_full_llm
Typed protocol: typed_role_contract_v1
Prompt revision: grounded_inputs_deliverables_single_block_v3
Model ablation: qwen35_9b_bnb_4bit
Model: lainlives/Qwen3.5-9B-bnb-4bit
Base model: Qwen/Qwen3.5-9B
Quantization: BNB_NF4_PREQUANTIZED
On-the-fly 4-bit quantization: False
Output dir: /content/tec_agent_project/outputs/metrics/experiment_5b_qwen35_9b_bnb_4bit
Selected tasks: ['hightec_midlat_europe', 'stable_midlat_europe', 'compare_midlat_europe_highlat_north', 'compare_three_regions', 'report_midlat_europe']


## Planned test questions

This preview is for the human notebook reader only. The expected tool sequence is used later for evaluation and is not passed into the typed LLM agents.

In [8]:
preview_rows = []
for config in SELECTED_TASK_CONFIGS:
    preview_rows.append({
        "preset_id": config["preset_id"],
        "task_type": config["task_type"],
        "query": config["query"],
        "expected_tool_sequence": " -> ".join(build_five_task_expected_sequence(config)),
    })
pd.DataFrame(preview_rows)

,preset_id,task_type,query,expected_tool_sequence
0,hightec_midlat_europe,high_tec,Find high TEC intervals for midlat_europe from...,tec_get_timeseries -> tec_compute_high_thresho...
1,stable_midlat_europe,stable_intervals,Find stable TEC intervals for midlat_europe fr...,tec_get_timeseries -> tec_compute_stability_th...
2,compare_midlat_europe_highlat_north,compare_regions,Compare TEC statistics for midlat_europe and h...,tec_get_timeseries -> tec_get_timeseries -> te...
3,compare_three_regions,compare_regions,Compare TEC statistics for equatorial_atlantic...,tec_get_timeseries -> tec_get_timeseries -> te...
4,report_midlat_europe,report,Build a concise TEC analysis report for midlat...,tec_get_timeseries -> tec_compute_series_stats...


## 6. Dataset setup

In [9]:
DATASET_PATH = PROCESSED_DATASET_PATH
assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"
register_dataset(dataset_ref=DATASET_REF, path=DATASET_PATH, file_format="parquet")
get_dataset_summary(DATASET_REF)

{'dataset_ref': 'default',
 'n_rows': 2184,
 'n_columns': 10,
 'region_columns': ['equatorial_atlantic',
  'equatorial_africa',
  'equatorial_pacific',
  'midlat_europe',
  'midlat_usa',
  'midlat_asia',
  'midlat_south_america',
  'midlat_australia',
  'highlat_north',
  'highlat_south'],
 'start': '2024-01-01 00:00:00',
 'end': '2024-03-31 23:00:00',
 'index_type': 'DatetimeIndex'}

## 7. Qwen3.5-9B-AWQ loading

The checkpoint is loaded directly as `QuantTrio/Qwen3.5-9B-AWQ`, using the image-text Transformers class advertised by the model card. This is a notebook-local adapter with the same `generate(messages, ...) -> str` interface expected by the existing typed runner; the Python agent implementation is not changed.


In [10]:
# def _resolve_dtype(dtype_name: str | None):
#     if dtype_name is None or dtype_name == "auto":
#         return "auto"
#     aliases = {
#         "float16": torch.float16,
#         "fp16": torch.float16,
#         "bfloat16": torch.bfloat16,
#         "bf16": torch.bfloat16,
#         "float32": torch.float32,
#         "fp32": torch.float32,
#     }
#     return aliases[dtype_name.lower()]


# def _apply_stop_strings(text: str, stop: list[str] | None) -> str:
#     if not stop:
#         return text
#     cut_at = None
#     for item in stop:
#         idx = text.find(item)
#         if idx >= 0 and (cut_at is None or idx < cut_at):
#             cut_at = idx
#     return text if cut_at is None else text[:cut_at].strip()


# class Qwen35AWQChatModel:
#     """Notebook-local text chat adapter for the pre-quantized Qwen3.5-9B-AWQ checkpoint."""

#     def __init__(
#         self,
#         model_name: str,
#         device_map: str = "auto",
#         torch_dtype: str | None = "float16",
#         trust_remote_code: bool = True,
#         max_input_tokens: int | None = None,
#     ):
#         if model_name != "QuantTrio/Qwen3.5-9B-AWQ":
#             raise ValueError(f"This ablation notebook must load only QuantTrio/Qwen3.5-9B-AWQ, got {model_name!r}")
#         self.model_name = model_name
#         self.max_input_tokens = max_input_tokens
#         self.processor = AutoProcessor.from_pretrained(
#             model_name,
#             trust_remote_code=trust_remote_code,
#         )
#         self.model = AutoModelForImageTextToText.from_pretrained(
#             model_name,
#             device_map=device_map,
#             torch_dtype=_resolve_dtype(torch_dtype),
#             low_cpu_mem_usage=True,
#             trust_remote_code=trust_remote_code,
#         ).eval()
#         if hasattr(self.model, "config"):
#             self.model.config.use_cache = True

#     def generate(
#         self,
#         messages: list[dict[str, str]],
#         max_new_tokens: int = 512,
#         temperature: float = 0.0,
#         do_sample: bool = False,
#         stop: list[str] | None = None,
#     ) -> str:
#         prompt_messages = self._format_messages(messages)
#         try:
#             inputs = self.processor.apply_chat_template(
#                 prompt_messages,
#                 tokenize=True,
#                 add_generation_prompt=True,
#                 return_dict=True,
#                 return_tensors="pt",
#                 enable_thinking=False,
#             )
#         except TypeError:
#             inputs = self.processor.apply_chat_template(
#                 prompt_messages,
#                 tokenize=True,
#                 add_generation_prompt=True,
#                 return_dict=True,
#                 return_tensors="pt",
#             )

#         if self.max_input_tokens is not None and inputs["input_ids"].shape[-1] > self.max_input_tokens:
#             keep = self.max_input_tokens
#             inputs["input_ids"] = inputs["input_ids"][..., -keep:]
#             if "attention_mask" in inputs:
#                 inputs["attention_mask"] = inputs["attention_mask"][..., -keep:]

#         device = self._input_device()
#         if device is not None:
#             inputs = inputs.to(device)

#         generation_kwargs = {
#             "max_new_tokens": max_new_tokens,
#             "do_sample": do_sample,
#             "use_cache": True,
#         }
#         tokenizer = getattr(self.processor, "tokenizer", None)
#         if tokenizer is not None and getattr(tokenizer, "eos_token_id", None) is not None:
#             generation_kwargs["pad_token_id"] = tokenizer.eos_token_id
#         if do_sample:
#             generation_kwargs["temperature"] = temperature

#         with torch.inference_mode():
#             output_ids = self.model.generate(**inputs, **generation_kwargs)

#         input_len = inputs["input_ids"].shape[-1]
#         text = self.processor.decode(
#             output_ids[0][input_len:],
#             skip_special_tokens=True,
#         ).strip()
#         return _apply_stop_strings(text, stop)

#     def _format_messages(self, messages: list[dict[str, str]]) -> list[dict[str, Any]]:
#         formatted = []
#         for message in messages:
#             content = message.get("content", "")
#             if isinstance(content, str):
#                 content = [{"type": "text", "text": content}]
#             formatted.append({"role": message.get("role", "user"), "content": content})
#         return formatted

#     def _input_device(self):
#         device = getattr(self.model, "device", None)
#         if device is not None:
#             return device
#         try:
#             return next(self.model.parameters()).device
#         except StopIteration:
#             return None


# model = Qwen35AWQChatModel(
#     model_name=MODEL_NAME,
#     device_map="auto",
#     torch_dtype=TORCH_DTYPE,
#     trust_remote_code=True,
#     max_input_tokens=MAX_INPUT_TOKENS,
# )
# print("Loaded AWQ model:", MODEL_NAME)


In [11]:
# import gc
# from typing import Any

# import torch
# from transformers import AutoModel, AutoModelForImageTextToText, AutoProcessor


def _resolve_dtype(dtype_name: str | None):
    if dtype_name is None or dtype_name == "auto":
        return "auto"

    aliases = {
        "float16": torch.float16,
        "fp16": torch.float16,
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
        "float32": torch.float32,
        "fp32": torch.float32,
    }

    key = dtype_name.lower()
    if key not in aliases:
        raise ValueError(
            f"Unsupported TORCH_DTYPE={dtype_name!r}. "
            f"Supported values: {sorted(aliases)} or 'auto'."
        )

    return aliases[key]


def _apply_stop_strings(text: str, stop: list[str] | None) -> str:
    if not stop:
        return text

    cut_at = None
    for item in stop:
        idx = text.find(item)
        if idx >= 0 and (cut_at is None or idx < cut_at):
            cut_at = idx

    return text if cut_at is None else text[:cut_at].strip()


class Qwen35BNB4BitChatModel:
    """
    Notebook-local text chat adapter for the pre-quantized
    Qwen3.5-9B bitsandbytes 4-bit checkpoint.

    This class loads an already quantized checkpoint from Hugging Face.
    It does NOT perform on-the-fly quantization.
    """

    EXPECTED_MODEL_NAME = "lainlives/Qwen3.5-9B-bnb-4bit"

    def __init__(
        self,
        model_name: str,
        device_map: str = "auto",
        dtype: str | None = "float16",
        trust_remote_code: bool = True,
        max_input_tokens: int | None = None,
    ):
        if model_name != self.EXPECTED_MODEL_NAME:
            raise ValueError(
                "This model-ablation notebook must load only "
                f"{self.EXPECTED_MODEL_NAME!r}, got {model_name!r}."
            )

        self.model_name = model_name
        self.max_input_tokens = max_input_tokens

        print("Loading processor:", model_name)
        self.processor = AutoProcessor.from_pretrained(
            model_name,
            trust_remote_code=trust_remote_code,
        )

        model_kwargs = {
            "device_map": device_map,
            "dtype": _resolve_dtype(dtype),
            "low_cpu_mem_usage": True,
            "trust_remote_code": trust_remote_code,
        }

        print("Loading pre-quantized BNB 4-bit model:", model_name)

        try:
            self.model = AutoModelForImageTextToText.from_pretrained(
                model_name,
                **model_kwargs,
            ).eval()
            self.model_loader_class = "AutoModelForImageTextToText"

        except (ValueError, TypeError, KeyError) as exc:
            # Some derivative Qwen3.5 checkpoints expose their custom generation
            # model through AutoModel instead of AutoModelForImageTextToText.
            # This fallback loads the SAME checkpoint and does not change the model.
            print(
                "AutoModelForImageTextToText loader was not accepted by this "
                "checkpoint. Retrying the same checkpoint through AutoModel."
            )
            print("Initial loader error:", repr(exc))

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            self.model = AutoModel.from_pretrained(
                model_name,
                **model_kwargs,
            ).eval()
            self.model_loader_class = "AutoModel"

        if not hasattr(self.model, "generate"):
            raise TypeError(
                f"Loaded model class {type(self.model).__name__!r} "
                "does not expose .generate(), so it cannot be used by the agent runner."
            )

        if hasattr(self.model, "config"):
            self.model.config.use_cache = True

        quantization_config = getattr(
            getattr(self.model, "config", None),
            "quantization_config",
            None,
        )

        print("Loaded model class:", type(self.model).__name__)
        print("Loader class:", self.model_loader_class)
        print("Quantization config:", quantization_config)

    def generate(
        self,
        messages: list[dict[str, str]],
        max_new_tokens: int = 512,
        temperature: float = 0.0,
        do_sample: bool = False,
        stop: list[str] | None = None,
    ) -> str:
        prompt_messages = self._format_messages(messages)

        template_kwargs = {
            "tokenize": True,
            "add_generation_prompt": True,
            "return_dict": True,
            "return_tensors": "pt",
        }

        try:
            inputs = self.processor.apply_chat_template(
                prompt_messages,
                enable_thinking=False,
                **template_kwargs,
            )
        except TypeError:
            inputs = self.processor.apply_chat_template(
                prompt_messages,
                **template_kwargs,
            )

        if (
            self.max_input_tokens is not None
            and inputs["input_ids"].shape[-1] > self.max_input_tokens
        ):
            keep = self.max_input_tokens
            inputs["input_ids"] = inputs["input_ids"][..., -keep:]

            if "attention_mask" in inputs:
                inputs["attention_mask"] = inputs["attention_mask"][..., -keep:]

        device = self._input_device()
        if device is not None:
            inputs = {
                key: value.to(device) if hasattr(value, "to") else value
                for key, value in inputs.items()
            }

        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": do_sample,
            "use_cache": True,
        }

        tokenizer = getattr(self.processor, "tokenizer", None)
        if tokenizer is not None and getattr(tokenizer, "eos_token_id", None) is not None:
            generation_kwargs["pad_token_id"] = tokenizer.eos_token_id

        if do_sample:
            generation_kwargs["temperature"] = temperature

        with torch.inference_mode():
            output_ids = self.model.generate(
                **inputs,
                **generation_kwargs,
            )

        input_len = inputs["input_ids"].shape[-1]

        text = self.processor.decode(
            output_ids[0][input_len:],
            skip_special_tokens=True,
        ).strip()

        return _apply_stop_strings(text, stop)

    @staticmethod
    def _format_messages(
        messages: list[dict[str, str]],
    ) -> list[dict[str, Any]]:
        formatted = []

        for message in messages:
            content = message.get("content", "")

            if isinstance(content, str):
                content = [{"type": "text", "text": content}]

            formatted.append(
                {
                    "role": message.get("role", "user"),
                    "content": content,
                }
            )

        return formatted

    def _input_device(self):
        device = getattr(self.model, "device", None)

        if device is not None and device.type != "meta":
            return device

        try:
            return next(
                parameter.device
                for parameter in self.model.parameters()
                if parameter.device.type != "meta"
            )
        except StopIteration:
            return None


model = Qwen35BNB4BitChatModel(
    model_name=MODEL_NAME,
    device_map="auto",
    dtype=TORCH_DTYPE,
    trust_remote_code=True,
    max_input_tokens=MAX_INPUT_TOKENS,
)

print("Loaded pre-quantized BNB 4-bit model:", MODEL_NAME)

Loading processor: lainlives/Qwen3.5-9B-bnb-4bit


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pre-quantized BNB 4-bit model: lainlives/Qwen3.5-9B-bnb-4bit


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/759 [00:00<?, ?it/s]

[transformers] Qwen3_5ForConditionalGeneration LOAD REPORT from: lainlives/Qwen3.5-9B-bnb-4bit
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model class: Qwen3_5ForConditionalGeneration
Loader class: AutoModelForImageTextToText
Quantization config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

Loaded pre-quantized BNB 4-bit model: lainlives/Qwen3.5-9B-bnb-4bit


## 8. Model load and GPU memory verification

Run this diagnostic before the five-task batch. If the pre-quantized AWQ checkpoint leaves too little free VRAM, the notebook stops before expensive agent inference.


In [12]:
def gpu_memory_gb():
    if not torch.cuda.is_available():
        return None
    props = torch.cuda.get_device_properties(0)
    total = props.total_memory / 1024**3
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    free_estimate = total - reserved
    return {
        "gpu": torch.cuda.get_device_name(0),
        "total_gb": total,
        "allocated_gb": allocated,
        "reserved_gb": reserved,
        "free_estimate_gb": free_estimate,
    }

quantization_config = getattr(getattr(model, "model", None), "config", None)
quantization_config = getattr(quantization_config, "quantization_config", None)
device_map = getattr(getattr(model, "model", None), "hf_device_map", None)

print("Model:", MODEL_NAME)
print("Base model metadata:", BASE_MODEL_NAME)
print("Quantization:", QUANTIZATION_FORMAT)
print("Model device map:", device_map)
print("Model dtype setting:", TORCH_DTYPE)
print("Model config quantization_config:", quantization_config)
print("MAX_INPUT_TOKENS:", MAX_INPUT_TOKENS)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
print("MAX_ORCHESTRATION_STEPS:", MAX_ORCHESTRATION_STEPS)
print("MAX_ROLE_STEPS:", MAX_ROLE_STEPS)

memory = gpu_memory_gb()
if memory is None:
    print("CUDA is not available. Colab GPU runtime is required for the actual batch run.")
else:
    print("GPU:", memory["gpu"])
    print("Total VRAM GB:", round(memory["total_gb"], 2))
    print("Allocated after model load GB:", round(memory["allocated_gb"], 2))
    print("Reserved after model load GB:", round(memory["reserved_gb"], 2))
    print("Estimated free VRAM GB:", round(memory["free_estimate_gb"], 2))
    if memory["free_estimate_gb"] < MIN_FREE_VRAM_GB_AFTER_LOAD:
        raise RuntimeError(
            "Model loaded, but remaining VRAM is too small for a reliable five-task agent run. "
            "Do not start batch inference; reduce generation/context settings or choose another checkpoint."
        )
print("Ready for short generation smoke test:", True)


Model: lainlives/Qwen3.5-9B-bnb-4bit
Base model metadata: Qwen/Qwen3.5-9B
Quantization: BNB_NF4_PREQUANTIZED
Model device map: None
Model dtype setting: bfloat16
Model config quantization_config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

MAX_INPUT_TOKENS: 4096
MAX_NEW_TOKENS: 512
MAX_ORCHESTRATION_STEPS: 12
MAX_ROLE_STEPS: 8
GPU: Tesla T4
Total VRAM GB: 14.56
Allocated after model load GB: 7.34
Reserved after model load GB: 7.36
Estimated free VRAM GB: 7.2
Ready for short generation smoke test: True


## 9. Single-generation smoke test

This checks only that the AWQ model can produce a short text response with the notebook-local adapter. It is not an agent run and is not included in experiment metrics.


In [13]:
smoke_messages = [
    {"role": "system", "content": "You are a concise technical assistant."},
    {"role": "user", "content": "Reply with exactly: ready"},
]
smoke_output = model.generate(
    smoke_messages,
    max_new_tokens=32,
    temperature=0.0,
    do_sample=False,
)
print("Smoke output:", smoke_output)
assert smoke_output.strip(), "Smoke generation returned an empty response."
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    memory_after_smoke = gpu_memory_gb()
    print("Reserved after smoke GB:", round(memory_after_smoke["reserved_gb"], 2))
    print("Estimated free after smoke GB:", round(memory_after_smoke["free_estimate_gb"], 2))


Smoke output: ποipherals滨湖 Eyes优秀作品 tourist中国文化 îns经理谆 رائع ਪPubisé merek丰富多样的完成率 ਪalisieren ਪTrying ਪTrying,eg丰富多样的追查Tryingываетсяывается enough Manus wf
Reserved after smoke GB: 7.36
Estimated free after smoke GB: 7.2


## 10. Helpers


In [14]:
def to_jsonable(value):
    if hasattr(value, "to_dict"):
        return to_jsonable(value.to_dict())
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if hasattr(value, "item"):
        return value.item()
    return value


def metric_dict(metric_result):
    if metric_result is None:
        return {}
    return metric_result.metrics if isinstance(metric_result, MetricResult) else dict(metric_result)


def final_answer_preview(result, limit=360):
    text = getattr(result, "answer", "") or ""
    return text[:limit] + ("..." if len(text) > limit else "")


def build_verdict_checks(result, metric_result):
    metrics = metric_dict(metric_result)
    return {
        "agent_success": bool(result.success),
        "tool_sequence_match": metrics.get("tool_sequence_match"),
        "final_answer_present": bool(result.answer),
        "no_parse_errors": result.parse_error_count == 0,
        "no_stall": not result.stalled_loop_detected,
        "no_forbidden_tool_calls": result.forbidden_tool_call_count == 0,
        "no_premature_role_completion": result.premature_role_completion_count == 0,
        "no_empty_analysis_findings": result.empty_findings_done_count == 0,
    }


def overall_ok_from_checks(checks):
    required = [v for v in checks.values() if isinstance(v, bool)]
    return bool(required) and all(required)


def key_metric_summary(task_type, metrics):
    if task_type == "high_tec":
        return f"threshold_abs_error={metrics.get('threshold_abs_error')}; interval_count_error={metrics.get('interval_count_error')}"
    if task_type == "stable_intervals":
        return f"stable_interval_count_error={metrics.get('stable_interval_count_error')}"
    if task_type == "compare_regions":
        return f"mean_abs_error_avg={metrics.get('mean_abs_error_avg')}; pairwise_delta_count_match={metrics.get('pairwise_delta_count_match')}"
    if task_type == "report":
        return f"required_artifacts_present={metrics.get('required_artifacts_present')}; grounded={metrics.get('report_grounded_in_artifacts')}"
    return ""


## 11. Batch loop


In [15]:
all_results = []
gold_runner = GoldRunner()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for task_config in SELECTED_TASK_CONFIGS:
    preset_id = task_config["preset_id"]
    query = task_config["query"]
    expected_sequence = build_five_task_expected_sequence(task_config)
    eval_task = build_five_task_eval_task(task_config)

    print()
    print("===", preset_id, "===")
    server = build_local_mcp_server(run_id=f"qwen_multi_agent_typed_v3_qwen35_9b_awq_{preset_id}_colab")
    client = LocalMCPClient(server)
    agent = LLMFullTypedMultiAgent(
        model=model,
        client=client,
        max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
        max_role_steps=MAX_ROLE_STEPS,
        max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
        max_parse_retries=MAX_PARSE_RETRIES,
        max_tool_retries=MAX_TOOL_RETRIES,
        temperature=TEMPERATURE,
        state_feedback_mode=STATE_FEEDBACK_MODE,
    )

    result = agent.run(query)
    result_dict = result.to_dict()

    gold = gold_runner.run(eval_task)
    metric_result = None
    gold_status = gold.status
    gold_error = gold.error
    gold_result = gold.result
    if gold.status == "success" and gold.result is not None:
        agent_metric_payload = dict(result.tool_results)
        agent_metric_payload.update({
            "parse_error_count": result.parse_error_count,
            "invalid_json_count": result.invalid_json_count,
            "invalid_role_protocol_count": result.invalid_role_protocol_count,
            "forbidden_tool_call_count": result.forbidden_tool_call_count,
            "repeated_tool_call_count": result.repeated_tool_call_count,
            "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
            "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
            "stalled_loop_detected": result.stalled_loop_detected,
        })
        metric_result = compare_agent_to_gold(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            agent_result=agent_metric_payload,
            gold_result=gold.result,
            agent_trace=result.trace,
            task=task_to_dict(eval_task),
            parsed_task=result.parsed_task,
            orchestration_steps=result.orchestration_steps,
        )

    metrics = metric_dict(metric_result)
    verdict_checks = build_verdict_checks(result, metric_result)
    overall_ok = overall_ok_from_checks(verdict_checks)

    record = {
        "selected_preset": preset_id,
        "task_config": task_config,
        "query": query,
        "expected_tool_sequence": expected_sequence,
        "actual_tool_sequence": result.actual_tool_sequence,
        "architecture": ARCHITECTURE_MODE,
        "model_name": MODEL_NAME,
        "base_model_name": BASE_MODEL_NAME,
        "quantization_format": QUANTIZATION_FORMAT,
        "model_ablation_id": MODEL_ABLATION_ID,
        "typed_protocol_version": TYPED_PROTOCOL_VERSION,
        "prompt_revision": PROMPT_REVISION,
        "agent_result": result_dict,
        "gold_status": gold_status,
        "gold_error": gold_error,
        "gold_result": gold_result,
        "metrics": metrics,
        "verdict_checks": verdict_checks,
        "overall_ok": overall_ok,
        "success": bool(result.success),
        "final_answer_preview": final_answer_preview(result),
        "role_agent_order": result.role_agent_order,
        "orchestrator_decisions": result.orchestrator_decisions,
        "role_actions": result.role_actions,
        "role_assignments": result.role_assignments,
        "role_outputs": result.role_outputs,
        "tool_observations": result.tool_observations,
        "available_artifacts_snapshots": result.available_artifacts_snapshots,
        "invalid_assignment_count": result.invalid_assignment_count,
        "invalid_role_action_count": result.invalid_role_action_count,
        "invalid_role_response_count": result.invalid_role_response_count,
        "invalid_final_answer_count": result.invalid_final_answer_count,
        "invalid_tool_name_count": result.invalid_tool_name_count,
        "forbidden_tool_call_count": result.forbidden_tool_call_count,
        "multiple_protocol_blocks_in_single_output_count": result.multiple_protocol_blocks_in_single_output_count,
        "invalid_artifact_handle_count": result.invalid_artifact_handle_count,
        "premature_role_completion_count": result.premature_role_completion_count,
        "empty_findings_done_count": result.empty_findings_done_count,
        "repeated_equivalent_role_assignment_count": result.repeated_equivalent_role_assignment_count,
        "tool_error_count": result.tool_error_count,
        "tool_schema_validation_error_count": result.tool_schema_validation_error_count,
        "invalid_role_protocol_count": result.invalid_role_protocol_count,
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "repeated_tool_call_count": result.repeated_tool_call_count,
        "stalled_loop_detected": result.stalled_loop_detected,
        "repair_attempt_count": result.repair_attempt_count,
        "retry_count": result.retry_count,
        "recovery_attempt_count": result.recovery_attempt_count,
        "recovery_success_count": result.recovery_success_count,
        "recovery_failure_count": result.recovery_failure_count,
        "key_metric_summary": key_metric_summary(task_config["task_type"], metrics),
        "missing_agent_terminal_artifact": (metrics or {}).get("missing_agent_terminal_artifact"),
    }

    per_task_path = OUTPUT_DIR / PER_TASK_OUTPUT_TEMPLATE.format(preset_id=preset_id)
    with per_task_path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(record), f, ensure_ascii=False, indent=2)
    print("Saved:", per_task_path)
    print("success:", record["success"], "overall_ok:", overall_ok, "tools:", result.actual_tool_sequence)
    all_results.append(record)



=== hightec_midlat_europe ===
Saved: /content/tec_agent_project/outputs/metrics/experiment_5b_qwen35_9b_bnb_4bit/qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_hightec_midlat_europe_colab.json
success: False overall_ok: False tools: []

=== stable_midlat_europe ===
Saved: /content/tec_agent_project/outputs/metrics/experiment_5b_qwen35_9b_bnb_4bit/qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_stable_midlat_europe_colab.json
success: False overall_ok: False tools: []

=== compare_midlat_europe_highlat_north ===
Saved: /content/tec_agent_project/outputs/metrics/experiment_5b_qwen35_9b_bnb_4bit/qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_compare_midlat_europe_highlat_north_colab.json
success: False overall_ok: False tools: []

=== compare_three_regions ===
Saved: /content/tec_agent_project/outputs/metrics/experiment_5b_qwen35_9b_bnb_4bit/qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_compare_three_regions_colab.json
success: False overall_ok: False tools: []

=== report_midlat_europe ===
Save

## 12. Summary table and save batch


In [16]:
def rate(values):
    values = [v for v in values if isinstance(v, bool)]
    return None if not values else sum(values) / len(values)

summary = {
    "n_tasks": len(all_results),
    "n_success": sum(1 for r in all_results if r["success"]),
    "n_failed": sum(1 for r in all_results if not r["success"]),
    "success_rate": rate([r["success"] for r in all_results]),
    "n_overall_ok": sum(1 for r in all_results if r["overall_ok"]),
    "overall_ok_rate": rate([r["overall_ok"] for r in all_results]),
    "agent_success_rate": rate([r["success"] for r in all_results]),
    "tool_sequence_match_rate": rate([(r.get("metrics") or {}).get("tool_sequence_match") for r in all_results]),
    "role_agent_order_match_rate": rate([(r.get("metrics") or {}).get("role_agent_order_match") for r in all_results]),
    "artifact_flow_valid_rate": rate([(r.get("metrics") or {}).get("artifact_flow_valid") for r in all_results]),
    "required_role_agents_called_rate": rate([(r.get("metrics") or {}).get("required_role_agents_called") for r in all_results]),
    "avg_tool_call_count": sum(len(r["actual_tool_sequence"]) for r in all_results) / max(1, len(all_results)),
    "avg_tool_error_count": sum(r["tool_error_count"] for r in all_results) / max(1, len(all_results)),
    "avg_parse_error_count": sum(r["parse_error_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_assignment_count": sum(r["invalid_assignment_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_role_response_count": sum(r["invalid_role_response_count"] for r in all_results) / max(1, len(all_results)),
    "avg_forbidden_tool_call_count": sum(r["forbidden_tool_call_count"] for r in all_results) / max(1, len(all_results)),
    "avg_multiple_protocol_blocks_in_single_output_count": sum(r["multiple_protocol_blocks_in_single_output_count"] for r in all_results) / max(1, len(all_results)),
    "avg_invalid_artifact_handle_count": sum(r["invalid_artifact_handle_count"] for r in all_results) / max(1, len(all_results)),
    "avg_premature_role_completion_count": sum(r["premature_role_completion_count"] for r in all_results) / max(1, len(all_results)),
    "avg_empty_findings_done_count": sum(r["empty_findings_done_count"] for r in all_results) / max(1, len(all_results)),
    "avg_repeated_equivalent_role_assignment_count": sum(r["repeated_equivalent_role_assignment_count"] for r in all_results) / max(1, len(all_results)),
    "avg_tool_schema_validation_error_count": sum(r["tool_schema_validation_error_count"] for r in all_results) / max(1, len(all_results)),
    "stalled_loop_rate": rate([r["stalled_loop_detected"] for r in all_results]),
    "legacy_report_tool_used_rate": rate([("tec_" + "build_report") in r["actual_tool_sequence"] for r in all_results]),
}

payload = {
    "experiment_id": EXPERIMENT_ID,
    "architecture": ARCHITECTURE_MODE,
    "model_name": MODEL_NAME,
    "base_model_name": BASE_MODEL_NAME,
    "quantization_format": QUANTIZATION_FORMAT,
    "model_ablation_id": MODEL_ABLATION_ID,
    "dataset_ref": DATASET_REF,
    "dataset_path": str(DATASET_PATH),
    "model_config": {
        "model_name": MODEL_NAME,
        "base_model_name": BASE_MODEL_NAME,
        "quantization_format": QUANTIZATION_FORMAT,
        "model_ablation_id": MODEL_ABLATION_ID,
        "load_in_4bit": LOAD_IN_4BIT,
        "load_in_8bit": LOAD_IN_8BIT,
        "torch_dtype": TORCH_DTYPE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": DO_SAMPLE,
    },
    "agent_config": {
        "max_orchestration_steps": MAX_ORCHESTRATION_STEPS,
        "max_role_steps": MAX_ROLE_STEPS,
        "max_tool_calls_per_role": MAX_TOOL_CALLS_PER_ROLE,
        "max_parse_retries": MAX_PARSE_RETRIES,
        "max_tool_retries": MAX_TOOL_RETRIES,
        "state_feedback_mode": STATE_FEEDBACK_MODE,
    },
    "typed_protocol_version": TYPED_PROTOCOL_VERSION,
    "prompt_revision": PROMPT_REVISION,
    "summary": summary,
    "results": all_results,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with BATCH_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(payload), f, ensure_ascii=False, indent=2)

summary_rows = []
for r in all_results:
    summary_rows.append({
        "preset_id": r["selected_preset"],
        "task_type": r["task_config"]["task_type"],
        "agent_success": r["success"],
        "overall_ok": r["overall_ok"],
        "tool_sequence_match": (r.get("metrics") or {}).get("tool_sequence_match"),
        "actual_tool_sequence": " -> ".join(r["actual_tool_sequence"]),
        "role_agent_order": " -> ".join(r["role_agent_order"]),
        "tool_call_count": len(r["actual_tool_sequence"]),
        "tool_error_count": r["tool_error_count"],
        "final_answer_present": bool((r["agent_result"] or {}).get("answer")),
        "role_agent_order_match": (r.get("metrics") or {}).get("role_agent_order_match"),
        "required_role_agents_called": (r.get("metrics") or {}).get("required_role_agents_called"),
        "artifact_flow_valid": (r.get("metrics") or {}).get("artifact_flow_valid"),
        "invalid_artifact_handle_count": r["invalid_artifact_handle_count"],
        "multiple_protocol_blocks_in_single_output_count": r["multiple_protocol_blocks_in_single_output_count"],
        "premature_role_completion_count": r["premature_role_completion_count"],
        "empty_findings_done_count": r["empty_findings_done_count"],
        "repeated_equivalent_role_assignment_count": r["repeated_equivalent_role_assignment_count"],
        "tool_schema_validation_error_count": r["tool_schema_validation_error_count"],
        "parse_error_count": r["parse_error_count"],
        "repeated_tool_call_count": r["repeated_tool_call_count"],
        "forbidden_tool_call_count": r["forbidden_tool_call_count"],
        "stalled_loop_detected": r["stalled_loop_detected"],
        "missing_agent_terminal_artifact": r.get("missing_agent_terminal_artifact"),
        "key_metric_summary": r["key_metric_summary"],
        "answer_preview": r["final_answer_preview"],
    })
print("Saved batch:", BATCH_OUTPUT_PATH)
pd.DataFrame(summary_rows)


Saved batch: /content/tec_agent_project/outputs/metrics/experiment_5b_qwen35_9b_bnb_4bit/qwen_multi_agent_typed_v3_qwen35_9b_bnb_4bit_batch_colab.json


,preset_id,task_type,agent_success,overall_ok,tool_sequence_match,actual_tool_sequence,role_agent_order,tool_call_count,tool_error_count,final_answer_present,...,empty_findings_done_count,repeated_equivalent_role_assignment_count,tool_schema_validation_error_count,parse_error_count,repeated_tool_call_count,forbidden_tool_call_count,stalled_loop_detected,missing_agent_terminal_artifact,key_metric_summary,answer_preview
0,hightec_midlat_europe,high_tec,False,False,False,,,0,0,False,...,0,0,0,3,0,0,True,high_intervals,threshold_abs_error=None; interval_count_error...,
1,stable_midlat_europe,stable_intervals,False,False,False,,,0,0,False,...,0,0,0,3,0,0,True,stable_intervals,stable_interval_count_error=None,
2,compare_midlat_europe_highlat_north,compare_regions,False,False,False,,,0,0,False,...,0,0,0,3,0,0,True,comparison_id,mean_abs_error_avg=None; pairwise_delta_count_...,
3,compare_three_regions,compare_regions,False,False,False,,,0,0,False,...,0,0,0,3,0,0,True,comparison_id,mean_abs_error_avg=None; pairwise_delta_count_...,
4,report_midlat_europe,report,False,False,False,,,0,0,False,...,0,0,0,3,0,0,True,report,required_artifacts_present=None; grounded=False,


## 13. Optional model-ablation comparison

This cell reads existing JSON files only. Missing sources are skipped gracefully. The central comparison is the same typed v3 architecture on `Qwen/Qwen3.5-4B` versus `QuantTrio/Qwen3.5-9B-AWQ`.


In [17]:
def load_optional(path):
    if not path.exists():
        print("Missing:", path)
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def records_by_preset(batch):
    if not batch:
        return {}
    return {r.get("selected_preset"): r for r in batch.get("results", [])}


def seq_text(record):
    if not record:
        return None
    return " -> ".join(record.get("actual_tool_sequence") or [])


def final_present(record):
    if not record:
        return None
    agent_result = record.get("agent_result") or {}
    return bool(agent_result.get("answer") or record.get("final_answer_preview"))


def summary_row(label, batch):
    if not batch:
        return None
    summary = batch.get("summary") or {}
    return {
        "model": label,
        "success_rate": summary.get("success_rate"),
        "overall_ok_rate": summary.get("overall_ok_rate"),
        "tool_sequence_match_rate": summary.get("tool_sequence_match_rate"),
        "role_order_match_rate": summary.get("role_agent_order_match_rate"),
        "artifact_flow_valid_rate": summary.get("artifact_flow_valid_rate"),
        "avg_tool_call_count": summary.get("avg_tool_call_count"),
        "avg_tool_error_count": summary.get("avg_tool_error_count"),
        "avg_invalid_artifact_handle_count": summary.get("avg_invalid_artifact_handle_count"),
        "stalled_loop_rate": summary.get("stalled_loop_rate"),
    }

comparison_sources = {
    "qwen_single": load_optional(OUTPUT_ROOT / "qwen_single_agent_batch_colab.json"),
    "rule_multi": load_optional(OUTPUT_ROOT / "multi_agent_rule_based_five_task_batch.json"),
    "qwen_multi_old": load_optional(OUTPUT_ROOT / "qwen_multi_agent_batch_colab.json"),
    "typed_v1_minimal_role_response": load_optional(OUTPUT_ROOT / "qwen_multi_agent_typed_batch_colab.json"),
    "typed_v2_role_boundaries_tool_schemas": load_optional(OUTPUT_ROOT / "experiment_3" / "qwen_multi_agent_typed_v2_batch_colab.json"),
    "typed_v3_qwen35_4b": load_optional(OUTPUT_ROOT / "experiment_4" / "qwen_multi_agent_typed_v3_batch_colab.json"),
    "typed_v3_qwen35_9b_awq": load_optional(BATCH_OUTPUT_PATH),
}

four_b = records_by_preset(comparison_sources["typed_v3_qwen35_4b"])
nine_b = records_by_preset(comparison_sources["typed_v3_qwen35_9b_awq"])

model_ablation_rows = []
for config in SELECTED_TASK_CONFIGS:
    preset_id = config["preset_id"]
    r4 = four_b.get(preset_id)
    r9 = nine_b.get(preset_id)
    model_ablation_rows.append({
        "preset": preset_id,
        "task_type": config["task_type"],
        "Qwen3.5-4B typed v3 success": None if r4 is None else r4.get("success"),
        "Qwen3.5-9B-AWQ typed v3 success": None if r9 is None else r9.get("success"),
        "4B tool sequence": seq_text(r4),
        "9B tool sequence": seq_text(r9),
        "4B stalled": None if r4 is None else r4.get("stalled_loop_detected"),
        "9B stalled": None if r9 is None else r9.get("stalled_loop_detected"),
        "4B invalid handles": None if r4 is None else r4.get("invalid_artifact_handle_count"),
        "9B invalid handles": None if r9 is None else r9.get("invalid_artifact_handle_count"),
        "final answer change": f"{final_present(r4)} -> {final_present(r9)}",
    })

aggregate_rows = [
    row for row in [
        summary_row("Qwen3.5-4B typed v3", comparison_sources["typed_v3_qwen35_4b"]),
        summary_row("Qwen3.5-9B-AWQ typed v3", comparison_sources["typed_v3_qwen35_9b_awq"]),
    ] if row is not None
]

print("Model-ablation per-task comparison")
display(pd.DataFrame(model_ablation_rows))
print("Model-ablation aggregate comparison")
display(pd.DataFrame(aggregate_rows))


Missing: /content/tec_agent_project/outputs/metrics/qwen_single_agent_batch_colab.json
Missing: /content/tec_agent_project/outputs/metrics/multi_agent_rule_based_five_task_batch.json
Missing: /content/tec_agent_project/outputs/metrics/qwen_multi_agent_batch_colab.json
Missing: /content/tec_agent_project/outputs/metrics/qwen_multi_agent_typed_batch_colab.json
Missing: /content/tec_agent_project/outputs/metrics/experiment_3/qwen_multi_agent_typed_v2_batch_colab.json
Missing: /content/tec_agent_project/outputs/metrics/experiment_4/qwen_multi_agent_typed_v3_batch_colab.json
Model-ablation per-task comparison


,preset,task_type,Qwen3.5-4B typed v3 success,Qwen3.5-9B-AWQ typed v3 success,4B tool sequence,9B tool sequence,4B stalled,9B stalled,4B invalid handles,9B invalid handles,final answer change
0,hightec_midlat_europe,high_tec,None,False,None,,None,True,None,0,None -> False
1,stable_midlat_europe,stable_intervals,None,False,None,,None,True,None,0,None -> False
2,compare_midlat_europe_highlat_north,compare_regions,None,False,None,,None,True,None,0,None -> False
3,compare_three_regions,compare_regions,None,False,None,,None,True,None,0,None -> False
4,report_midlat_europe,report,None,False,None,,None,True,None,0,None -> False


Model-ablation aggregate comparison


,model,success_rate,overall_ok_rate,tool_sequence_match_rate,role_order_match_rate,artifact_flow_valid_rate,avg_tool_call_count,avg_tool_error_count,avg_invalid_artifact_handle_count,stalled_loop_rate
0,Qwen3.5-9B-AWQ typed v3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
